# Project 3 — Advanced LangChain Research Assistant

## Overview

Build an advanced research assistant using **LangChain** that can search a knowledge base, retrieve relevant information, use tools, and generate grounded answers.

The project will gradually extend a basic RAG pipeline into a more capable LangChain application.

## What We Will Learn

- Semantic Chunking
- Advanced Retrieval
- Reranking
- Prompt Engineering
- Structured Output
- Tool Calling
- Conversation History
- Memory
- Streaming
- Error Handling
- Retries and Fallbacks
- Agents
- LangSmith
- Evaluation

## High-Level Architecture

```text
Documents
    ↓
Document Loading
    ↓
Advanced Chunking
    ↓
Embeddings
    ↓
Vector Store
    ↓
Retriever
    ↓
Reranking
    ↓
Context
    ↓
LLM
    ↓
Tools / Agent
    ↓
Final Answer
    ↓
Evaluation & LangSmith

### ***Step 1 — Project Setup***

In [ ]:
### Load the Environment variables
import os
from dotenv import load_dotenv
load_dotenv()

#### ***Step 2 — Load the Research Documents***

In [ ]:
### valid the Document Path

path = "03_Project/Data/Documents"

if os.path.exists(path):
    print("Path is Existed")

else:
    print("Path is not Existed")

In [ ]:
### load the all files using Directory Loader
from langchain_community.document_loaders import DirectoryLoader,TextLoader

### Initialize the loader
loader = DirectoryLoader(
    path,
    glob = "*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
print("Number Of Documents:",len(documents))

In [ ]:
for doc in documents:
    print("Metadata:",doc.metadata)
    print("Content:",doc.page_content)
    print("#"*60)

#### ***Step 3: Semantic Chunking***
#### **Splits the Document based on semantic meaning, instead of fixed size**

In [ ]:
### For Semantic Chunking we have initialize the embedding model first
### Create a Embedding model by HuggingFace with 1024 dimensions
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

In [ ]:
#semantic chunker
from langchain_experimental.text_splitter import SemanticChunker
semantic_splitter = SemanticChunker(
    embedding_model,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount = 90
)

In [ ]:
#chunks means splits the documents into smaller parts
chunks = semantic_splitter.split_documents(documents)
print("Number Of Chunks:",len(chunks))

In [ ]:
## Few chunks are empty let's handle it
clean_chunks  = [chunk for chunk in chunks if chunk.page_content.strip()]
len(clean_chunks)

In [ ]:
for i,chunk in enumerate(clean_chunks,start=1):
    print("="*60)
    print(f"CHUNK {i}")
    print("="*60)
    print("Content:",chunk.page_content)

#### ***Step 3: Create the vector store***

In [ ]:
###Using Chroma to store and create vectors
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents = clean_chunks,
    embedding = embedding_model,
    persist_directory = "../../vectorstore/research_assistant_rag",
    collection_name = "research_assistant_rag"
)


In [ ]:
### Let's see how many vectors was created
vectorstore._collection.count()

#### ***Step 4 — Similarity Search.***

In [ ]:
## retriever
similarity_retriever = vectorstore.as_retriever(

    search_type = "similarity",
    search_kwargs = {"k":2}
)

In [ ]:
## test queries how similarity search works
test_queries = [
    "What is LangChain?",
    "What are the core abstractions in LangChain?",
    "What is an embedding?",
    "What is a vector database?",
    "How does semantic search work?",
    "What is hybrid retrieval?",
    "What is an AI agent?",
    "How do agents use tools?",
    "What is tool calling?",
    "What is an LLM?",
    "What are important LLM concepts?",
    "What is prompt engineering?",
    "What makes a good prompt?",
    "What is Retrieval-Augmented Generation?",
    "How does RAG work?",
    "What is reranking in RAG?",
    "What does the retrieval parameter k control?",
    "Why is retrieval quality important in RAG?"
]
for query in test_queries:
    retrieved_docs = similarity_retriever.invoke(query)
    print("="*60)
    print("Query:",query)
    print("="*60)

    for i,doc in enumerate(retrieved_docs,1):
        print(f"------ Document {i}-------")
        print("Metadata:",doc.metadata)
        print("Page Content:",doc.page_content)

In [ ]:
negative_queries = [
    "What are the ICU visiting hours?",
    "What is the cardiologist's phone number?",
    "What is the patient's date of birth?"
]
for query in negative_queries:
    retrieved_docs = similarity_retriever.invoke(query)
    print("="*60)
    print("Query:",query)
    print("="*60)

    for i,doc in enumerate(retrieved_docs,1):
        print(f"------ Document {i}-------")
        print("Metadata:",doc.metadata)
        print("Page Content:",doc.page_content)

#### ***Reranker***
#### **Find Potentially Relevant Documents**

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

In [ ]:
## Retrieve the top chunks after performing the reranking
def retriever_rerank(query,k=5):

    retrieved_docs = similarity_retriever.invoke(query)[:k]

    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    top_chunks = [doc for(doc,score) in ranked_docs[:2]]

    return top_chunks

In [ ]:
results = retriever_rerank("What is semantic search?")
results

In [ ]:
### format the context

def build_context(top_chunks):

    return "\n\n".join(doc.page_content for doc in top_chunks)

In [ ]:
chat_history = []

#### ***Prompt***

In [ ]:
#### design a prompt
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_template("""
    Answer the following question based on the provided context only.

    if you don't find the answer based on the question,
    say I don't have enough information based on the provided context.
    Question:{question}

    Context:{context}
""")

#### ***LLM***


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)
llm

#### ***Step 5 — Build the RAG chain***

In [ ]:
### output parser
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
rag_chain = ({
    "context":retriever_rerank | RunnableLambda(build_context),
    "question":RunnablePassthrough()
}
| prompt
| llm
| output_parser)



In [ ]:
query = "What is semantic search?"
response = rag_chain.invoke(query)
response

#### **RAG Evaluation**

In [ ]:
test_queries = [
    "What is semantic search?",
    "What is an embedding?",
    "What is RAG?",
    "What is an agent?",
    "What is the cardiologist's phone number?"
]

In [ ]:
for query in test_queries:
    response = rag_chain.invoke(query)
    print("="*70)
    print("Query:",query)
    print("="*70)
    print(response)

In [ ]:
from langchain_core.tools import tool

@tool
def rag_response(query: str) -> str:
    """Search the Research Assistant's local knowledge base and answer questions using the retrieved documents.

    ALWAYS try this tool FIRST for any conceptual or definitional question that could plausibly be
    covered by the local knowledge base (e.g. AI/ML terms, RAG, embeddings, vector databases, agents,
    tool calling, prompt engineering, LLMs, semantic search, etc.). Only fall back to the web/news
    search tools if this tool's answer says there isn't enough information, or if the question is
    clearly about something time-sensitive (news, releases, prices) that a static knowledge base
    would not contain.

    Args:
        query: The user's question, in natural language.

    Returns:
        A grounded answer based on the retrieved document chunks, or a note that there isn't
        enough information in the knowledge base.
    """

    response = rag_chain.invoke(query)

    return response

In [ ]:
from langchain_classic.tools import DuckDuckGoSearchRun

### Single shared search client, reused by both websearch_tool and news_search_tool
_ddg_search = DuckDuckGoSearchRun()

@tool
def websearch_tool(query: str) -> str:
    """
    Search the live web for general current information that is NOT breaking news.

    Use this tool for things like:
    - The current state of a product, technology, or company (e.g. "latest LangChain features")
    - Documentation or technical developments that may postdate the model's knowledge cutoff
    - Any fact lookup that isn't in the local knowledge base and isn't a news headline

    Do NOT use this for breaking news or "what's happening today" questions — use
    news_search_tool for those instead. Base your answer only on facts that actually appear
    in the returned results; do not invent numbers, dates, or quotes that aren't present there.

    Args:
        query: A clear and specific search query.

    Returns:
        Relevant information retrieved from the live web.
    """

    return _ddg_search.invoke(query)

In [ ]:
@tool
def news_search_tool(query: str) -> str:
    """
    Search the live web specifically for breaking news headlines and recent events.

    Use this tool ONLY when the user explicitly asks about news, headlines, or "what's
    happening today/this week" — i.e. discrete, dated events reported by news outlets.
    For general product/technology updates that aren't framed as news, use websearch_tool
    instead.

    Base your answer only on facts, figures, and quotes that actually appear in the search
    results. If the results are thin or vague, say so rather than filling in plausible-sounding
    specifics.

    Args:
        query: A clear and specific news search query.

    Returns:
        Relevant recent news information from the web.
    """

    return _ddg_search.invoke(f"latest news: {query}")

In [ ]:
### Summarize Prompt
summarize_prompt = ChatPromptTemplate.from_template("""
        You are an Helpful AI Assistant.

        Please Summarize the provided context in a meaningful way without extra information.

        Context:{context}
""")

summarize_chain = summarize_prompt | llm |output_parser

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

@tool
def fetch_url(url: str) -> str:
    """Fetch the content from a URL and return a concise summary of it.

    Use this only when the user provides a specific URL and asks you to fetch, summarize,
    or explain it. Only summarize what is actually present on the fetched page; do not add
    outside information.
    """
    try:
        loader = WebBaseLoader(url)
        documents = loader.load()
    except Exception as e:
        return f"Could not fetch the URL '{url}': {e}"

    if not documents:
        return f"No content could be extracted from '{url}'."

    context = "\n\n".join(doc.page_content for doc in documents)
    response = summarize_chain.invoke({"context": context})

    return response

In [ ]:
tools = [rag_response,websearch_tool,news_search_tool,fetch_url]

llm_with_tools = llm.bind_tools(tools)

## Tool Map
tool_map = {tool.name:tool for tool in tools}

tool_map

In [ ]:
### Tool-use policy for the agent — controls routing between the knowledge base and web/news search,
### and enforces grounding so the model doesn't invent details beyond what tools actually return.

SYSTEM_PROMPT = """You are a research assistant with access to tools. Follow this policy strictly:

1. For any conceptual or definitional question that could plausibly be covered by the local
   knowledge base (AI/ML concepts, RAG, embeddings, agents, tool calling, prompt engineering,
   etc.), ALWAYS call `rag_response` first before considering any other tool.
2. Only use `websearch_tool` or `news_search_tool` when the question is clearly time-sensitive,
   or when `rag_response` indicates it doesn't have enough information.
3. Use `news_search_tool` only for breaking news / "what's happening today" questions.
   Use `websearch_tool` for other current-information lookups.
4. Use `fetch_url` only when the user provides a specific URL.
5. When you use any search or fetch tool, base your answer ONLY on what the tool result actually
   contains. Do not invent specific numbers, dates, quotes, or named entities that are not present
   in the tool output. If the tool result is thin, say so explicitly rather than filling in
   plausible-sounding details.
"""

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

def tool_loop_execution(query, max_iterations=6):
    messages = [SystemMessage(SYSTEM_PROMPT), HumanMessage(query)]

    for _ in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']

            tool = tool_map[tool_name]
            print("Tool Name:", tool)

            try:
                result = tool.invoke(tool_args)
            except Exception as e:
                result = f"Error while running tool '{tool_name}': {e}"

            messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))

    # Exceeded max_iterations without a final answer — return the last response we got
    return response

#### ***RAG Tool***

In [ ]:
rag_queries = [
    "What is semantic search?",
    "What are the latest developments in LangChain?",
    "What are the latest AI news today?",
    "Summarize this webpage: https://www.langchain.com/"
]

for query in rag_queries:
    print("="*60)
    print("Query:",query)
    print("="*60)
    response = tool_loop_execution(query).content
    print("Response:",response)